## 1.0 Libraries and directories

In [27]:
import ee 
import geemap
import geopandas as gpd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

## 2.0 Select an ROI

In [2]:
rois = ee.data.listAssets('projects/alpod-412314/assets/ROIs/')
target_name = 'YKflats'
target_roi = None

for roi in rois['assets']:
    if roi['id'] == f'projects/alpod-412314/assets/ROIs/{target_name}_roi':
        target_roi = roi
        
roi = ee.FeatureCollection(target_roi['id']).first()
roi = roi.geometry()

## 3.0 Select target dates

In [18]:
date = '2020-05-29'
date_plus1d = '2020-05-30'

def get_img_roi_intersection(img, roi, scale, extract_number):
    """ Returns polygons for images intersection with roi """
    data_mask = img.mask().reduce(ee.Reducer.anyNonZero())
    img_boundary = data_mask.reduceToVectors(
        geometry=roi,
        geometryType='polygon',
        scale=scale,
        maxPixels=1e13
    )
    polygon = ee.Feature(img_boundary.toList(img_boundary.size()).get(extract_number))

    return polygon


# Sentinel-2
def find_s2(roi, date, date_plus1d):
    """ Creates a mosiac for Sentinel-2 on given dates"""

    s2_img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(date, date_plus1d))

    def rescale_s2(img):
        """ Rescaled Sentinel-2 (except SCL band) to match LandSat """
        bands = ['B2', 'B3', 'B4', 'B8', 'B8A']
        rescale_bands = img.select(bands).divide(10000)
        other_bands = img.select('SCL')
        rescaled_img = rescale_bands.addBands(other_bands)

        return rescaled_img

    s2_img = (s2_img.select(['B2', 'B3', 'B4', 'B8', 'B8A'])
              .mosaic()
              .clip(roi))

    s2_img = rescale_s2(s2_img)
    s2_extent = get_img_roi_intersection(s2_img, roi, 10, 0)

    return s2_img, s2_extent

s2_img, s2_extent = find_s2(roi, date, date_plus1d)

def find_ls(roi, date, date_plus1d):
    """ Recales Landsat images"""
    ls_img = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(roi)
          .filterDate(date, date_plus1d))

    def ls_rescale(img):
        img = img.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])
        # Converts digital numbers to surface reflectance
        img = img.multiply(0.0000275).add(-0.2)

        return img

    ls_img = ls_img.mosaic().clip(roi)
    ls_img = ls_rescale(ls_img)
    ls_extent = get_img_roi_intersection(ls_img, roi, 30, 2)

    return ls_img, ls_extent

ls_img, ls_extent = find_ls(roi, date, date_plus1d)

# s2_true_col_params = {
#     'bands': ['B4', 'B3', 'B2'],
#     'min': 0,
#     'max': 0.3,
#     'gamma': 1.7
# }
ls_true_col_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.7
}

# Map = geemap.Map()
# Map.addLayer(s2_img, s2_true_col_params, 'S2 True Color')
# Map.addLayer(ls_img, ls_true_col_params, 'LS True Color')
# Map.addLayer(roi, {'color': 'red'}, 'ROI Outline')
# Map.addLayer(ls_extent, {'color': 'green'}, 'LS Extent')
# Map.addLayer(s2_extent, {'color': 'blue'}, 'S2 Extent')
# Map.centerObject(roi, zoom=10)
# Map

## 4.0 Clip both image to a common footprint

In [34]:
s2_extent_geom = s2_extent.geometry()
ls_extent_geom = ls_extent.geometry()
img_overlap = ls_extent_geom.intersection(s2_extent_geom, maxError=ee.ErrorMargin(10))

del s2_extent_geom, ls_extent_geom

ls_img = ls_img.clip(img_overlap)
s2_img = s2_img.clip(img_overlap)

# Estimate the utm zone of image overlap
def calculate_utm_zone(polygon):
    centroid = polygon.centroid()
    lon = centroid.coordinates().getNumber(0)
    lat = centroid.coordinates().getNumber(1)
    
    utm_zone = lon.add(180).divide(6).floor().mod(60).add(1)

    is_northern = True
    epsg_code = ee.Number(
        ee.Algorithms.If(
            is_northern,
            utm_zone.add(32600)))
    
    return epsg_code

est_utm = calculate_utm_zone(img_overlap)
pp.pp(est_utm.getInfo())

32606


## 5.0 Get a common cloud mask

In [36]:
def get_ls_mask(bounds, date, date_plus1d):
    """Generates a cloud mask for Landsat 8 images using QA_PIXEL bit flags."""
    ls_qa = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
             .filterBounds(bounds)
             .filterDate(date, date_plus1d)
             .select('QA_PIXEL')
             .mosaic()
             .clip(bounds))

    # Define bitmasks for the conditions
    cloudBitMask = 1 << 3        # Bit 3: Cloud
    cloudShadowBitMask = 1 << 4  # Bit 4: Cloud Shadow
    snowBitMask = 1 << 5         # Bit 5: Snow
    cirrusBitMask = 1 << 2       # Bit 2: Cirrus
    dilatedCloudBitMask = 1 << 1 # Bit 1: Dilated Cloud

    # Combine all bitmasks into one
    bitmask = (cloudBitMask
               | cloudShadowBitMask
               | snowBitMask
               | cirrusBitMask
               | dilatedCloudBitMask)

    # Create the mask where any of the bits are set
    ls_full_mask = ls_qa.bitwiseAnd(bitmask).neq(0)

    return ls_full_mask

def get_s2_mask(bounds, date, date_plus1d):

    s2_clouds = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(bounds)
    .filterDate(date, date_plus1d)
    .mosaic().
    .clip(bounds))

    s2_scl = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .select('SCL')
    .filterBounds(bounds)
    .filterDate(date, date_plus1d)
    .mosaic().
    .clip(bounds))

    clouds_binary = s2_clouds.select('probability').gt(15).rename('cl_binary')
    s2_dark_mask = s2_scl.eq(2)
    s2_cloud_shaddow_mask = s2_scl.eq(3)
    s2_cirrus_mask = s2_scl.eq(10) 

    s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)

    # Resample the sentinel-2 mask to Landsat resolution??
    

ls_mask = get_ls_mask(img_overlap, date, date_plus1d)
s2_mask = get_s2_mask(img_overlap, date, date_plus1d)

In [38]:
binary_mask_red_params = {
    'min':0,
    'max':1,
    'palette': ['grey', 'red']
}

binary_mask_blue_params = {
    'min':0,
    'max':1,
    'palette': ['grey', 'blue']
}


Map = geemap.Map()
Map.addLayer(ls_img, ls_true_col_params, 'LS True Color')
Map.addLayer(ls_mask, binary_mask_red_params, 'LS Mask')
Map.centerObject(roi, zoom=10)
Map

Map(center=[66.5185907221393, -146.00018537875334], controls=(WidgetControl(options=['position', 'transparent_…